# Day 5 · Making it count — rules, real mortality, and what you cannot claim

**Today's goal:** convert a discovered cluster into a rule the underwriting engine can
evaluate, test the groups against the actual `death` table, and be precise about the causal
claims you are *not* making.

### Summary

A pattern that cannot be written as a rule and does not separate an outcome is an
interesting plot, not a deliverable. Today closes both gaps. The regulatory constraint from
the review is the reason: ML cannot be used live in the underwriting decision, so the output
has to be a **generic rule**, not a per-patient score.

### Technique chosen per subtopic

| Subtopic | We use | Mentioned only |
|---|---|---|
| Rule extraction | **shallow decision tree surrogate** | RuleFit, skope-rules, Bayesian rule lists |
| Anomaly detection | **isolation forest** | one-class SVM, LOF, autoencoder reconstruction |
| Outcome validation | **Kaplan-Meier + log-rank**, implemented | Cox PH, random survival forest, competing risks |
| Feature attribution | *mentioned* | SHAP, permutation importance |
| Causal discovery | *mentioned* | PC/FCI, NOTEARS, invariant causal prediction |

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 150); pd.set_option('display.max_columns', 40)
import ethos_omop as eo

OUT_DIR = "../data_input"   # extract_cohort() output; use "omop_extract" for extract_omop()
DOMAINS = ['Visit', 'Condition', 'Drug', 'Procedure', 'Measurement']

if not os.path.isdir(OUT_DIR):
    raise FileNotFoundError(
        f"{OUT_DIR}/ not found. Run Day 1's eo.extract_cohort(...) (or "
        f"eo.extract_omop(...) with OUT_DIR='omop_extract') first, or "
        f"python make_demo_extract.py for a synthetic stand-in.")

concept  = eo.load_concept(OUT_DIR)
codesets = eo.build_codesets(concept, verbose=False)
events   = eo.tag_events(eo.load_events(OUT_DIR, domains=DOMAINS), codesets)
person   = eo.load_person(OUT_DIR)
truth    = eo.load_truth(OUT_DIR)                   # None against real data

print(f"events {events.shape}   patients {events.person_id.nunique()}")
print(f"codeset coverage: {events.in_codeset.mean():.1%} of events carry a clinical tag")
print(f"truth labels: {'present (DEMO extract)' if truth is not None else 'ABSENT (real data)'}")
events.head(6)

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans

X   = eo.bag_of_codes(events, min_count=20)
Z   = TruncatedSVD(min(24, X.shape[1] - 1), random_state=0).fit_transform(
          normalize(X.values.astype(float)))
lab = KMeans(7, n_init=10, random_state=0).fit(Z).labels_
print(pd.Series(lab).value_counts().sort_index().to_string())

## 1 · Cluster to rule — the surrogate tree

Fit a deliberately shallow tree predicting *cluster membership* from raw claim features. The
tree is not the model; it is a readable approximation of the boundary the clustering drew.
Cross-validated F1 tells you how much you lose by simplifying.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import cross_val_score

def cluster_rule(c, depth=3, leaf=30):
    tgt  = (lab == c).astype(int)
    tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                                  class_weight='balanced', random_state=0)
    f1 = cross_val_score(tree, X.values, tgt, cv=5, scoring='f1').mean()
    tree.fit(X.values, tgt)
    return tree, f1

fid = []
for c in range(7):
    _, f1 = cluster_rule(c)
    top = (X.groupby(lab).mean().loc[c] / (X.mean() + .01)).sort_values(ascending=False)
    fid.append({'cluster': c, 'n': int((lab == c).sum()), 'surrogate_F1': round(f1, 3),
                'top_tags': ', '.join(top.head(3).index)})
display(pd.DataFrame(fid).sort_values('surrogate_F1', ascending=False))

In [ ]:
C = int(pd.DataFrame(fid).sort_values('surrogate_F1', ascending=False).cluster.iloc[0])
tree, f1 = cluster_rule(C, depth=3)
print(f"cluster {C}: surrogate F1 = {f1:.3f}\n")
print(export_text(tree, feature_names=list(X.columns), max_depth=3))

Read the tree as a predicate — something close to:

```
IF cad OR (troponin AND NOT cad)
THEN cohort = 'cardiac_workup_positive'
```

Writable, testable, explainable to a regulator. **Low surrogate F1 is itself a finding**: the
cluster has no simple description and should not become a rule. Report that rather than
shipping a rule you cannot state.

In [ ]:
plt.figure(figsize=(13, 5))
plot_tree(tree, feature_names=list(X.columns), class_names=['other', f'cluster {C}'],
          filled=True, fontsize=7, impurity=False, proportion=True)
plt.title(f'surrogate rule for cluster {C}'); plt.tight_layout(); plt.show()

## 2 · Anomaly detection

Unusual trajectories are underwriting-interesting on their own, and outliers distort
clustering if left in.

In [ ]:
from sklearn.ensemble import IsolationForest

out = IsolationForest(contamination=0.03, random_state=0).fit_predict(Z) == -1
print("flagged:", out.sum())
print(pd.DataFrame({'outlier': out, 'n_tags': X.sum(axis=1).values})
        .groupby('outlier').n_tags.describe()[['count', 'mean', '50%', 'max']].round(1))
print("\nIf 'outlier' just means 'high utiliser', that is nuisance signal.")
print("Normalise by lookback and encounter count before believing it.")

## 3 · Real mortality from the `death` table

This is the part the synthetic version could not do. `survival_frame()` joins the extracted
`death` table, anchors time at each patient's first observed event, and administratively
censors at the extract date.

In [ ]:
death = eo.load_death(OUT_DIR)
sf    = eo.survival_frame(events, person, death).set_index('person_id').reindex(X.index)
sf['cluster'] = lab

print(sf.groupby('cluster')[['time_months', 'event']].agg(
      {'time_months': 'median', 'event': ['sum', 'mean']}).round(3))
print("\nLow event counts mean WIDE intervals, not a null result. Say so.")

In [ ]:
def kaplan_meier(time, event):
    order = np.argsort(time)
    t, e  = np.asarray(time)[order], np.asarray(event)[order]
    n, s  = len(t), 1.0
    ts, ss = [0.0], [1.0]
    for i in range(n):
        if e[i] == 1:
            s *= 1 - 1 / (n - i)
            ts.append(t[i]); ss.append(s)
    return np.array(ts), np.array(ss)

plt.figure(figsize=(8.5, 5))
rows = []
for c in range(7):
    m = sf.cluster == c
    ts, ss = kaplan_meier(sf.time_months[m].values, sf.event[m].values)
    plt.step(ts, ss, where='post', label=f'cluster {c} (n={m.sum()}, d={int(sf.event[m].sum())})')
    s60 = ss[ts <= 60][-1] if (ts <= 60).any() else 1.0
    rows.append({'cluster': c, 'n': int(m.sum()),
                 'deaths': int(sf.event[m].sum()), 'S(60mo)': round(s60, 3)})
plt.xlim(0, 120); plt.xlabel('months since first observed event'); plt.ylabel('survival')
plt.legend(fontsize=8); plt.title('Kaplan-Meier by discovered cluster (real death table)')
plt.tight_layout(); plt.show()

surv = pd.DataFrame(rows).sort_values('S(60mo)')
display(surv)

In [ ]:
from scipy.stats import chi2 as chi2dist

def logrank(t1, e1, t2, e2):
    t = np.concatenate([t1, t2]); e = np.concatenate([e1, e2])
    g = np.concatenate([np.zeros(len(t1)), np.ones(len(t2))])
    o = np.argsort(t); t, e, g = t[o], e[o], g[o]
    n1, n2 = len(t1), len(t2)
    O1 = E1 = V = 0.0
    for i in range(len(t)):
        n = n1 + n2
        if n <= 1:
            break
        if e[i] == 1:
            O1 += (g[i] == 0); E1 += n1 / n; V += (n1 * n2) / (n * n)
        if g[i] == 0: n1 -= 1
        else:         n2 -= 1
    if V <= 0:
        return np.nan, np.nan
    chi2 = (O1 - E1) ** 2 / V
    return chi2, 1 - chi2dist.cdf(chi2, 1)

worst, best = int(surv.cluster.iloc[0]), int(surv.cluster.iloc[-1])
m1, m2 = sf.cluster == worst, sf.cluster == best
chi2, p = logrank(sf.time_months[m1].values, sf.event[m1].values,
                  sf.time_months[m2].values, sf.event[m2].values)
print(f"cluster {worst} vs {best}:   chi2 = {chi2:.1f}   p = {p:.2e}")

Now the sentence has all three parts:

> *"Cluster 2, describable as the rule `cad OR troponin`, covers N patients and shows
> materially worse survival at 60 months."*

Pattern, rule, and outcome. That is the deliverable.

**Two honest caveats specific to this extract.** Death capture in claims is usually
incomplete, so absolute survival is biased upward — only the *contrast* between clusters is
interpretable. And time is anchored at first observed event, which is an enrolment artifact
rather than a clinical milestone; index on a clinical anchor instead when the question
allows it.

## 4 · What you cannot claim

The most important section, and the shortest.

**Patterns reflect care delivery as much as biology.** A trajectory mixes disease
progression, prescribing habits, formulary design, and — specific to Ethos — **your own
ordering rules**. Whether DX or BOLS gets pulled depends on what the applicant disclosed, so
a discovered pattern can be an artifact of the very underwriting workflow that generated the
data. That objection from the review is correct, and it is testable.

In [ ]:
# Confounding check: does the pattern's prevalence drift with enrolment year?
first_yr = events.groupby('person_id').event_date.min().dt.year.reindex(X.index)
chk = pd.DataFrame({'cluster': lab, 'first_year': first_yr.values})
tab = pd.crosstab(chk.first_year, chk.cluster, normalize='index').round(3)
display(tab.tail(8))
print("Sharp year-over-year shifts in a cluster's share suggest a DATA-COLLECTION")
print("artifact, not a clinical phenotype. Check before presenting anything.")

Three questions to answer before presenting any pattern:

1. **Could this exist because of how data is collected?** Compare prevalence across
   application years, products, and channels — the cell above is the first cut.
2. **Is the signal utilisation volume in disguise?** Control for `lookback_days` and
   encounter count.
3. **Would it appear in an external cohort?** You cannot know until the third-party
   de-identified data arrives. Say so explicitly.

Causal structure discovery (PC, FCI, NOTEARS) exists, and knowing it exists is mainly useful
so you can explain precisely why you are *not* claiming causality.

## 5 · The reporting template

| Field | Example |
|---|---|
| Name | `CARDIAC_WORKUP_POSITIVE` |
| Motif | `chest_pain -> troponin -> coronary_angio -> statin` |
| Prevalence | 8.6% of cohort |
| Stability | bootstrap ARI 0.79 |
| Surrogate rule | `cad OR troponin` |
| Surrogate fidelity | F1 0.79 |
| Outcome separation | S(60mo) contrast, log-rank p |
| Deaths observed | 31 of 171 — wide intervals |
| Known confounders | ED-channel applicants over-represented; year drift checked |
| Lookback restriction | >= 730 days |
| Clinical review | pending |

A blank row means the pattern is not ready.

## Exercises

1. Run the surrogate tree at depths 2, 3 and 5. Plot fidelity against depth — where is the
   interpretability knee?
2. Build rules from the Day 3 **motifs** instead of the clusters. Better rules? (Prediction:
   yes, because they are already sequential.)
3. Restrict to `lookback_days >= 730` and redo the whole pipeline. Which conclusions change?
4. Fill the reporting template end to end for one cluster. That single page is what goes to
   Zack.

## Where to go after this week

| Next | Why |
|---|---|
| SPMF library (Java) | the industrial catalog of sequential mining algorithms |
| `pm4py` | proper process mining, readable process models |
| Jensen et al. 2014, *Nature Communications* | temporal disease trajectories from registry data |
| Molnar, *Interpretable ML* (free) | the rule-extraction chapter |
| `meds_etl_omop` + `femr` | the CLMBR plumbing sprint |
| `lifelines`, `scikit-survival` | once the 25M-life mortality data lands |